# Notebook 4: ML Forecasting & Anomaly Detection

This notebook tests whether the teleconnection signals identified in Notebook 3 translate into operational forecast skill for compound wind hydro energy droughts.

**Sub question:** Which forecasting paradigm is most appropriate for climate teleconnection prediction: regularised linear methods, tree based ensembles, boosted ensembles, or sequence models?

**Structure:**
1. Feature engineering based on Notebook 3 findings
2. Walk forward cross validation with nested hyperparameter tuning
3. Four model comparison (ElasticNet, Random Forest, XGBoost, LSTM)
4. Feature group comparison (climate only vs local only vs combined)
5. Feature importance analysis
6. Anomaly detection

**Key inputs from Notebook 3:**
- SAM is the primary compound driver (DJF/MAM, lags 1-4)
- IOD is the strongest individual teleconnection (JJA runoff, lags 2-4 and 8-11)
- ONI is hydro-specific (JJA/SON, lags 0-3)
- Season is essential context (SAM reverses sign between DJF and JJA)
- SAM-WHDI teleconnection strength varies on decadal timescales

## Imports and Data Loading

In [3]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Models
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

df = pd.read_csv(
    "../data/processed/whdi_timeseries.csv", index_col="date", parse_dates=True
)

with open("../results/tables/feature_decisions.json", "r") as f:
    feature_decisions = json.load(f)

print(f"Data loaded: {df.shape}")
print(f"Period: {df.index[0]} to {df.index[-1]}")
print(
    f"\nPrimary target: {feature_decisions['primary_target']['variable']} "
    f"at {feature_decisions['primary_target']['lead_time']}-month lead"
)
print(
    f"Secondary target: {feature_decisions['secondary_target']['variable']} "
    f"at {feature_decisions['secondary_target']['lead_time']}-month lead"
)

Data loaded: (564, 13)
Period: 1979-01-01 00:00:00 to 2025-12-01 00:00:00

Primary target: whdi_3 at 3-month lead
Secondary target: whdi_6 at 3-month lead


## Feature Engineering

In [4]:
def create_lagged_features(df, feature_decisions):
    """
    Create the full feature matrix based on the teleconnection findings.

    Returns a dataframe with Group A (climate modes), Group B (local variables), and target variables.
    """
    results = pd.DataFrame(index=df.index)

    # Group A: Climate mode features

    # SAM at lags 1-4
    sam_lags = feature_decisions["features"]["sam"]["lags"]
    for lag in sam_lags:
        results[f"sam_lag{lag}"] = df["sam"].shift(lag)

    # ONI at lags 0-3
    oni_lags = feature_decisions["features"]["oni"]["lags"]
    for lag in oni_lags:
        results[f"oni_lag{lag}"] = df["oni"].shift(lag)

    # IOD at lags 2-4 and 8-11
    iod_lags = feature_decisions["features"]["iod"]["lags"]
    for lag in iod_lags:
        results[f"iod_lag{lag}"] = df["iod"].shift(lag)

    # Season one hot encoding
    # SON is dropped to avoid perfect multicollinearity
    results["season_DJF"] = df.index.month.isin([12, 1, 2]).astype(int)
    results["season_MAM"] = df.index.month.isin([3, 4, 5]).astype(int)
    results["season_JJA"] = df.index.month.isin([6, 7, 8]).astype(int)

    # Group B: Local variable features

    # WHDI3 at t, t-1, t-2
    results["whdi3_lag0"] = df["whdi_3"]
    results["whdi3_lag1"] = df["whdi_3"].shift(1)
    results["whdi3_lag2"] = df["whdi_3"].shift(2)

    # Component anomalies at t
    results["wind_std_lag0"] = df["wind_std"]
    results["runoff_std_lag0"] = df["runoff_std"]

    # Snowmelt anomaly at t (standardise by month)
    snowmelt_monthly_mean = df["snowmelt"].groupby(df.index.month).transform("mean")
    snowmelt_monthly_std = df["snowmelt"].groupby(df.index.month).transform("std")
    results["snowmelt_std_lag0"] = (
        df["snowmelt"] - snowmelt_monthly_mean
    ) / snowmelt_monthly_std

    # Target Variables

    lead = feature_decisions["primary_target"]["lead_time"]
    results["target_whdi3"] = df["whdi_3"].shift(-lead)
    results["target_whdi6"] = df["whdi_6"].shift(-lead)

    return results


# Create featueres
df_features = create_lagged_features(df, feature_decisions)

df_features = df_features.dropna()

print(f"Feature matrix: {df_features.shape}")
print(f"Period after lagging: {df_features.index[0]} to {df_features.index[-1]}")
print(f"Usable months: {len(df_features)}")

# Define feature groups
group_a_cols = [
    c for c in df_features.columns if c.startswith(("sam_", "oni_", "iod_", "season_"))
]
group_b_cols = [
    c
    for c in df_features.columns
    if c.startswith(("whdi3_", "wind_", "runoff_", "snowmelt_", "season_"))
]
all_feature_cols = list(set(group_a_cols + group_b_cols))
target_cols = ["target_whdi3", "target_whdi6"]

print(f"\nGroup A (climate modes): {len(group_a_cols)} features")
print(f"  {group_a_cols}")
print(f"\nGroup B (local conditions): {len(group_b_cols)} features")
print(f"  {group_b_cols}")
print(f"\nCombined: {len(all_feature_cols)} features")

Feature matrix: (547, 26)
Period after lagging: 1979-12-01 00:00:00 to 2025-06-01 00:00:00
Usable months: 547

Group A (climate modes): 18 features
  ['sam_lag1', 'sam_lag2', 'sam_lag3', 'sam_lag4', 'oni_lag0', 'oni_lag1', 'oni_lag2', 'oni_lag3', 'iod_lag2', 'iod_lag3', 'iod_lag4', 'iod_lag8', 'iod_lag9', 'iod_lag10', 'iod_lag11', 'season_DJF', 'season_MAM', 'season_JJA']

Group B (local conditions): 9 features
  ['season_DJF', 'season_MAM', 'season_JJA', 'whdi3_lag0', 'whdi3_lag1', 'whdi3_lag2', 'wind_std_lag0', 'runoff_std_lag0', 'snowmelt_std_lag0']

Combined: 24 features


## Walk Forward CV Setup

In [5]:
def create_walk_forward_folds(df_features, min_train_years=22, test_years=3):
    """
    Create expanding window walk forward cross validaiton folds.

    Each fold has an expanding training window and a fixed size test window.
    """
    folds = []

    years = sorted(df_features.index.year.unique())
    start_year = years[0]

    # First test period starts after the end of the training window (min_train_years)
    first_test_year = start_year + min_train_years

    fold_num = 1
    test_start = first_test_year

    while test_start + test_years <= years[-1] + 1:
        train_mask = df_features.index.year < test_start
        test_mask = (df_features.index.year >= test_start) & (
            df_features.index.year < test_start + test_years
        )

        train_idx = df_features.index[train_mask]
        test_idx = df_features.index[test_mask]

        if len(train_idx) > 0 and len(test_idx) > 0:
            folds.append(
                {
                    "fold": fold_num,
                    "train_start": train_idx[0].year,
                    "train_end": train_idx[-1].year,
                    "test_start": test_idx[0].year,
                    "test_end": test_idx[-1].year,
                    "n_train": len(train_idx),
                    "n_test": len(test_idx),
                    "train_idx": train_idx,
                    "test_idx": test_idx,
                }
            )
            fold_num += 1

        test_start += test_years

    return folds


folds = create_walk_forward_folds(df_features)

print(" " * 20 + "Walk Forward Cross Validation Folds")
print("=" * 75)
for fold in folds:
    print(
        f"  Fold {fold['fold']}: Train {fold['train_start']}-{fold['train_end']} "
        f"({fold['n_train']} months) | "
        f"Test {fold['test_start']}-{fold['test_end']} "
        f"({fold['n_test']} months)"
    )

                    Walk Forward Cross Validation Folds
  Fold 1: Train 1979-2000 (253 months) | Test 2001-2003 (36 months)
  Fold 2: Train 1979-2003 (289 months) | Test 2004-2006 (36 months)
  Fold 3: Train 1979-2006 (325 months) | Test 2007-2009 (36 months)
  Fold 4: Train 1979-2009 (361 months) | Test 2010-2012 (36 months)
  Fold 5: Train 1979-2012 (397 months) | Test 2013-2015 (36 months)
  Fold 6: Train 1979-2015 (433 months) | Test 2016-2018 (36 months)
  Fold 7: Train 1979-2018 (469 months) | Test 2019-2021 (36 months)
  Fold 8: Train 1979-2021 (505 months) | Test 2022-2024 (36 months)


## Hyperparameter Grids

In [6]:
# Define hyperparameter grids for each model
# Focused on regularisation parameters most relevant to small datasets

param_grids = {
    "elasticnet": {
        "alpha": [0.01, 0.1, 1.0, 10.0],
        "l1_ratio": [0.1, 0.5, 0.9],
    },
    "random_forest": {
        "n_estimators": [100, 300],
        "max_depth": [3, 5, 10, None],
        "min_samples_leaf": [5, 10, 20],
    },
    "xgboost": {
        "n_estimators": [100, 300],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.7, 1.0],
        "reg_alpha": [0, 0.1],
    },
}

# LSTM Config
LSTM_EPOCHS = 100
LSTM_PATIENCE = 10

lstm_config = [
    {"hidden_size": 32, "dropout": 0.2, "lr": 0.001, "seq_length": 6},
    {"hidden_size": 32, "dropout": 0.3, "lr": 0.0005, "seq_length": 12},
    {"hidden_size": 64, "dropout": 0.2, "lr": 0.001, "seq_length": 12},
    {"hidden_size": 64, "dropout": 0.3, "lr": 0.001, "seq_length": 6},
]

print("Hyperparameter search spaces")
for model, grid in param_grids.items():
    total = 1
    for v in grid.values():
        total *= len(v)
    print(f"    {model}: {total} combinations")
print(f"    lstm: {len(lstm_config)} predefined combinations")

Hyperparameter search spaces
    elasticnet: 12 combinations
    random_forest: 24 combinations
    xgboost: 48 combinations
    lstm: 4 predefined combinations


## Model Training Functions

In [7]:
def tune_sklearn_model(model_class, param_grid, X_train, y_train):
    """
    Tune a sklearn model using inner time series CV on training data.
    Returns the best fitted model and best parameters.
    """
    inner_cv = TimeSeriesSplit(n_splits=3)

    search = GridSearchCV(
        model_class,
        param_grid,
        cv=inner_cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        refit=True,
    )

    search.fit(X_train, y_train)

    return search.best_estimator_, search.best_params_, search.best_score_


def evaluate_predictions(y_true, y_pred):
    """Compute all evaluation metrics."""
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }


def compute_baselines(y_train, y_test, whdi_current):
    """
    Compute baseline predictions.

    Climatology: predict the training mean (=0 for standardised data)
    Persistence: predict current value as the future value
    """
    climatology_pred = np.full(len(y_test), y_train.mean())
    persistence_pred = whdi_current

    return {
        "climatology": climatology_pred,
        "persistence": persistence_pred,
    }


def skill_score(model_rmse, baseline_rmse):
    """Compute skill score: 1 - (model_rmse / baseline_rmse)"""
    if baseline_rmse == 0:
        return np.nan
    return 1 - (model_rmse / baseline_rmse) ** 2

## LSTM Model Definition

Due to the small amount of data, I have to keep the LSTM fairly simple so that it doesn't just memorise the training data and then fail on the test data.

In [8]:
class LSTMDroughtModel(nn.Module):
    """
    Simple single layer LSTM for drought prediction.

    Input: Sequence of monthly feature vectors
    Output: Single WHDI prediction
    """

    def __init__(self, input_size, hidden_size, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            dropout=0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape batch, seq_len, features
        lstm_out, (h_n, c_n) = self.lstm(x)
        # use last hidden state
        last_hidden = h_n.squeeze(0)
        dropped = self.dropout(last_hidden)
        output = self.fc(dropped)
        return output.squeeze(-1)


def create_lstm_sequence(X, seq_length):
    """
    Reshape flat feature matrix into overlapping sequence for LSTM.

    Each sample becomes a sequence of the previous seq_length months.
    """
    sequences = []
    for i in range(seq_length, len(X)):
        sequences.append(X[i - seq_length : i])
    return np.array(sequences)


def train_lstm(
    X_train_seq,
    y_train_seq,
    X_val_seq,
    y_val_seq,
    config,
    input_size,
    max_epochs,
    patience,
):
    """
    Train LSTM with early stopping on validation loss.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = LSTMDroughtModel(
        input_size=input_size,
        hidden_size=config["hidden_size"],
        dropout=config["dropout"],
    ).to(device)

    optimiser = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.MSELoss()

    # Convert to tensors
    X_train_t = torch.FloatTensor(X_train_seq).to(device)
    y_train_t = torch.FloatTensor(y_train_seq).to(device)
    X_val_t = torch.FloatTensor(X_val_seq).to(device)
    y_val_t = torch.FloatTensor(y_val_seq).to(device)

    # Training loop with early stopping
    best_val_loss = float("inf")
    best_model_state = None
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        optimiser.zero_grad()
        y_pred = model(X_train_t)
        loss = criterion(y_pred, y_train_t)
        loss.backward()
        optimiser.step()

        # Validation
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = criterion(val_pred, y_val_t).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Load best model
    model.load_state_dict(best_model_state)
    model.eval()

    return model, device

## Model Training Loop

In [ ]:
# This array will be changed based on testing how far into the future I want to see if it can predict, starting with 1-6 as the initial lead time is t+3,
# and the compute time is high
lead_times = [1, 2, 3, 4, 5, 6]

results = []

for lead in lead_times:
    print("\n" + "=" * 61)
    print(" " * 21 + f"Lead time: {lead} months")
    print("=" * 61)

    # Rebuild features and folds for the lead time
    feature_decisions["primary_target"]["lead_time"] = lead
    df_features = create_lagged_features(df, feature_decisions)
    df_features = df_features.dropna()
    folds = create_walk_forward_folds(df_features)

    for target_name, target_col in [
        ("whdi_3", "target_whdi3"),
        ("whdi_6", "target_whdi6"),
    ]:
        print(f"\nTarget: {target_name} at {lead} month lead")

        for fold in folds:
            print(
                f"\n  Fold {fold['fold']}: Train {fold['train_start']}-{fold['train_end']} | Test {fold['test_start']}-{fold['test_end']}"
            )

            # Split data
            train_data = df_features.loc[fold["train_idx"]]
            test_data = df_features.loc[fold["test_idx"]]

            # For each feature group configuration
            for group_name, feature_cols in [
                ("climate_only", group_a_cols),
                ("local_only", group_b_cols),
                ("combined", all_feature_cols),
            ]:
                X_train = train_data[feature_cols].values
                y_train = train_data[target_col].values
                X_test = test_data[feature_cols].values
                y_test = test_data[target_col].values

                # Scale features
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                # Persistence baseline
                if target_name == "whdi_3":
                    persistence_col = "whdi3_lag0"
                else:
                    persistence_col = "whdi3_lag0"

                whdi_current_test = (
                    test_data[persistence_col].values
                    if persistence_col in test_data.columns
                    else y_train[-len(y_test) :]
                )
                baselines = compute_baselines(y_train, y_test, whdi_current_test)

                for baseline_name, baseline_pred in baselines.items():
                    metrics = evaluate_predictions(y_test, baseline_pred)
                    results.append(
                        {
                            "lead": lead,
                            "target": target_name,
                            "fold": fold["fold"],
                            "feature_group": group_name,
                            "model": baseline_name,
                            "rmse": metrics["rmse"],
                            "mae": metrics["mae"],
                            "r2": metrics["r2"],
                            "best_params": None,
                        }
                    )

                # ELASTICNET
                try:
                    best_en, best_params_en, inner_score = tune_sklearn_model(
                        ElasticNet(max_iter=10000),
                        param_grids["elasticnet"],
                        X_train_scaled,
                        y_train,
                    )
                    en_pred = best_en.predict(X_test_scaled)
                    en_metrics = evaluate_predictions(y_test, en_pred)

                    results.append(
                        {
                            "lead": lead,
                            "target": target_name,
                            "fold": fold["fold"],
                            "feature_group": group_name,
                            "model": "elasticnet",
                            "rmse": en_metrics["rmse"],
                            "mae": en_metrics["mae"],
                            "r2": en_metrics["r2"],
                            "best_params": str(best_params_en),
                            "predictions": en_pred,
                        }
                    )
                    print(
                        f"    {group_name:15s} Elastic Net: RMSE={en_metrics['rmse']:.4f} params={best_params_en}"
                    )
                except Exception as e:
                    print(f"    {group_name:15s} Elastic Net: Failed {e}")

                # RANDOM FOREST
                try:
                    best_rf, best_params_rf, inner_score = tune_sklearn_model(
                        RandomForestRegressor(random_state=15),
                        param_grids["random_forest"],
                        X_train_scaled,
                        y_train,
                    )
                    rf_pred = best_rf.predict(X_test_scaled)
                    rf_metrics = evaluate_predictions(y_test, rf_pred)

                    results.append(
                        {
                            "lead": lead,
                            "target": target_name,
                            "fold": fold["fold"],
                            "feature_group": group_name,
                            "model": "random_forest",
                            "rmse": rf_metrics["rmse"],
                            "mae": rf_metrics["mae"],
                            "r2": rf_metrics["r2"],
                            "best_params": str(best_params_rf),
                            "predictions": rf_pred,
                        }
                    )
                    print(
                        f"    {group_name:15s} Random Forest: RMSE={rf_metrics['rmse']:.4f} params={best_params_rf}"
                    )
                except Exception as e:
                    print(f"    {group_name:15s} Random Forest: Failed {e}")

                # XGBOOST
                try:
                    best_xg, best_params_xg, inner_score = tune_sklearn_model(
                        xgb.XGBRegressor(random_state=15, verbosity=0),
                        param_grids["xgboost"],
                        X_train_scaled,
                        y_train,
                    )
                    xg_pred = best_xg.predict(X_test_scaled)
                    xg_metrics = evaluate_predictions(y_test, xg_pred)

                    results.append(
                        {
                            "lead": lead,
                            "target": target_name,
                            "fold": fold["fold"],
                            "feature_group": group_name,
                            "model": "xgboost",
                            "rmse": xg_metrics["rmse"],
                            "mae": xg_metrics["mae"],
                            "r2": xg_metrics["r2"],
                            "best_params": str(best_params_xg),
                            "predictions": xg_pred,
                        }
                    )
                    print(
                        f"    {group_name:15s} XGBoost: RMSE={xg_metrics['rmse']:.4f} params={best_params_xg}"
                    )
                except Exception as e:
                    print(f"    {group_name:15s} XGBoost: Failed {e}")

                # LSTM
                try:
                    best_lstm_val = float("inf")
                    best_lstm_model = None
                    best_lstm_config = None
                    best_lstm_device = None
                    best_lstm_test_seq = None
                    best_lstm_y_test = None

                    for config in lstm_config:
                        seq_len = config["seq_length"]

                        # Create sequences with this config
                        X_train_seq = create_lstm_sequence(X_train_scaled, seq_len)
                        y_train_seq = y_train[seq_len:].copy()

                        X_test_seq = create_lstm_sequence(
                            np.vstack([X_train_scaled[-seq_len:], X_test_scaled]),
                            seq_len,
                        )
                        y_test_seq = y_test.copy()

                        min_len = min(len(X_test_seq), len(y_test_seq))
                        X_test_seq = X_test_seq[:min_len]
                        y_test_seq = y_test_seq[:min_len]

                        # Validation Split
                        val_size = max(int(len(X_train_seq) * 0.2), seq_len)
                        X_val_seq = X_train_seq[-val_size:]
                        y_val_seq = y_train_seq[-val_size:]
                        X_train_lstm = X_train_seq[:-val_size]
                        y_train_lstm = y_train_seq[:-val_size]

                        model, device = train_lstm(
                            X_train_lstm,
                            y_train_lstm,
                            X_val_seq,
                            y_val_seq,
                            config,
                            input_size=X_train_scaled.shape[1],
                            max_epochs=LSTM_EPOCHS,
                            patience=LSTM_PATIENCE,
                        )

                        with torch.no_grad():
                            val_pred = model(torch.FloatTensor(X_val_seq).to(device))
                            val_loss = nn.MSELoss()(
                                val_pred,
                                torch.FloatTensor(y_val_seq).to(device),
                            ).to(device)

                        if val_loss < best_lstm_val:
                            best_lstm_val = val_loss
                            best_lstm_model = model
                            best_lstm_config = config
                            best_lstm_device = device
                            best_lstm_test_seq = X_test_seq
                            best_lstm_y_test = y_test_seq

                    # Predict with best config
                    with torch.no_grad():
                        lstm_pred = (
                            best_lstm_model(
                                torch.FloatTensor(best_lstm_test_seq).to(
                                    best_lstm_device
                                )
                            )
                            .cpu()
                            .numpy()
                        )

                    lstm_metrics = evaluate_predictions(best_lstm_y_test, lstm_pred)

                    results.append(
                        {
                            "lead": lead,
                            "target": target_name,
                            "fold": fold["fold"],
                            "feature_group": group_name,
                            "model": "lstm",
                            "rmse": lstm_metrics["rmse"],
                            "mae": lstm_metrics["mae"],
                            "r2": lstm_metrics["r2"],
                            "best_params": str(best_lstm_config),
                            "predictions": lstm_pred,
                        }
                    )
                    print(
                        f"    {group_name:15s} LSTM: RMSE={lstm_metrics['rmse']:.4f} params={best_lstm_config}"
                    )
                except Exception as e:
                    print(f"    {group_name:15s} LSTM: Failed {e}")

# Convert to datframe
df_results = pd.DataFrame(results)
df_results.to_csv("../results/tables/model_performance.csv", index=False)
print(f"\nAll results saved: {len(df_results)} rows")


                     Lead time: 1 months

Target: whdi_3 at 1 month lead

  Fold 1: Train 1979-2000 | Test 2001-2003 ---
    climate_only    Elastic Net: RMSE=1.4412 params={'alpha': 0.1, 'l1_ratio': 0.9}
    climate_only    Random Forest: RMSE=1.5510 params={'max_depth': 5, 'min_samples_leaf': 20, 'n_estimators': 100}
    climate_only    XGBoost: RMSE=1.5178 params={'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300, 'reg_alpha': 0, 'subsample': 0.7}
    climate_only    LSTM: RMSE=1.5274 params={'hidden_size': 32, 'dropout': 0.2, 'lr': 0.001, 'seq_length': 6}
    local_only      Elastic Net: RMSE=0.5797 params={'alpha': 0.01, 'l1_ratio': 0.1}
    local_only      Random Forest: RMSE=0.7140 params={'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 300}
    local_only      XGBoost: RMSE=0.7402 params={'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300, 'reg_alpha': 0, 'subsample': 0.7}
    local_only      LSTM: RMSE=1.2435 params={'hidden_size': 64, 'dropout': 0.3,